# Deep Agents 101: Journal Agent

This notebook builds one agent, one step at a time. Slides cover the concepts (harness, tools, HITL); this notebook covers the implementation.

For topics not covered today, see the self-paced LangChain Academy Deep Agents course.

## Setup: connect a model

**What you'll do:** install the SDKs and provide a model key.

In [ ]:
%pip install -q deepagents langchain-openai langgraph

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key (no cost to obtain, at openrouter.ai): ")

# To use an Anthropic or OpenAI key instead, uncomment:
# os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# To use a different key, replace the block above, for example:
# from langchain.chat_models import init_chat_model
# model = init_chat_model("anthropic:claude-haiku-4-5")

## 1: The harness, what you get before writing any tool code

<img src="./images/deepAgentsHarnessOverview.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Deep Agents harness overview">

**What you'll learn:** what filesystem read/write and planning capability a deep agent already has, with zero custom code.

**The payoff:** knowing the default baseline prevents rebuilding capabilities that already ship by default.

Every deep agent starts the same way: a model wrapped in a harness that already knows how to read and write files, plan, and call tools. No tools are added yet. The next cell shows what it can already do.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=model)

result = agent.invoke({"messages": [{"role": "user", "content": (
    "Start a journal.md file. Log a new dated entry from these notes:\n"
    "- What I learned today: ___\n"
    "- How I felt: ___\n"
    "- What's next: ___\n"
    "Then read the file back to me."
)}]})
print(result["messages"][-1].content)

**Next:** replace the three `___` placeholders above with your own notes, then rerun the cell.

## 2: Set its role with a system prompt

**What you'll learn:** how one `system_prompt` string controls the voice the agent writes in, on top of whatever facts you give it.

There is no system prompt yet. The `system_prompt` value passed to the agent is the entire prompt sent to the model. 

The agent still writes the entry itself: it takes your notes as raw facts and composes them into sentences (based on the persona you give it).

Run the cell below once as written, with no system prompt. Then uncomment one line (or write your own) and rerun with the same notes. The facts stay the same each time; only the voice the entry is written in changes.

In [ ]:
system_prompt = ""  # no system prompt yet, run once before making changes

# Then uncomment one of these (or write your own) and rerun:
# system_prompt = "You are a pirate. Answer only in pirate speak."
# system_prompt = "You are a toddler. Explain everything like you're five."
# system_prompt = "You are a melodramatic Victorian child. Narrate everything with excessive despair and flowery, dramatic language."

agent = create_deep_agent(model=model, system_prompt=system_prompt)

# Replace the ___ placeholders below with your own notes
result = agent.invoke({"messages": [{"role": "user", "content":
    "Log a three-sentence journal entry from these notes: what I learned today (___), "
    "how I felt (___), what's next (___)."
}]})
print(result["messages"][-1].content)

## 3: Give it a custom tool

**What you'll learn:** how a plain Python function becomes a tool the agent can call, either picked from these below or written yourself.

**The payoff:** tools are how an agent's abilities grow past reading and writing files, this is the piece you'll customize most often.

In [ ]:
import re
from collections import Counter
from langchain_core.tools import tool

@tool
def word_count(text: str) -> str:
    """Count the words in a piece of text."""
    return f"{len(text.split())} words"

@tool
def summarize_length(text: str, max_sentences: int = 2) -> str:
    """Trim a piece of text down to its first `max_sentences` sentences."""
    sentences = re.split(r"(?<=[.!?]) +", text.strip())
    return " ".join(sentences[:max_sentences])

@tool
def estimated_reading_time(text: str) -> str:
    """Estimate how long a piece of text takes to read, at 200 words per minute."""
    minutes = max(1, round(len(text.split()) / 200))
    return f"~{minutes} min read"

@tool
def keyword_extractor(text: str, top_n: int = 3) -> str:
    """Pull out the most frequent meaningful words in a piece of text."""
    stopwords = {"the", "a", "an", "and", "or", "but", "to", "of", "in", "on",
                 "for", "with", "i", "is", "it", "was", "my", "that", "this"}
    words = [w.strip(".,!?'\"").lower() for w in text.split()]
    counts = Counter(w for w in words if w not in stopwords and len(w) > 2)
    top = [word for word, _ in counts.most_common(top_n)]
    return ", ".join(top) if top else "no keywords found"

@tool
def mood_tag(text: str) -> str:
    """Tag a journal entry with a one-word mood."""
    # Placeholder: this always returns the same canned string, it does not
    # actually read the text. A real version needs an actual sentiment
    # analysis tool, for example:
    #   - nltk's VADER: SentimentIntensityAnalyzer().polarity_scores(text)
    #   - textblob: TextBlob(text).sentiment.polarity
    # both require downloading a lexicon/corpus at runtime, which is why
    # this demo keeps it as a stub instead.
    return f"(placeholder mood tag for '{text[:30]}...', replace with a real sentiment lookup)"

TOOL_MENU = {
    "word_count": word_count,
    "summarize_length": summarize_length,
    "estimated_reading_time": estimated_reading_time,
    "keyword_extractor": keyword_extractor,
    "mood_tag": mood_tag,
}

**Anatomy of a tool**: `@tool` turns a plain function into something the model can call:
- the **docstring** becomes the tool's description (how the model decides when to use it)
- the **type hints** become its input schema (what arguments it expects)
- the **return value** becomes what the model sees back

To write your own tool instead of using the menu, add it below.

In [ ]:
# TODO: pick from TOOL_MENU, or write your own @tool function
chosen_tool = TOOL_MENU["word_count"]

agent = create_deep_agent(model=model, system_prompt=system_prompt, tools=[chosen_tool])

result = agent.invoke({"messages": [{"role": "user", "content":
    "Here is a journal entry: 'Today I finally shipped the feature I've been stuck on for a "
    "week. The bug turned out to be a caching issue that took forever to track down, and I "
    "ended up rewriting most of the retry logic to fix it. It feels good to have it done, "
    "though I'm a little worried about whether the fix will hold up under real traffic. "
    "Tomorrow I want to write better tests before touching anything else.' "
    "Use your tool on it, then tell me what you found."
}]})
print(result["messages"][-1].content)

## 4: Human-in-the-loop, approve a risky action before it happens

<img src="./images/HITL.png" style="width: auto; height: 389px; border-radius: 8px;" alt="Human-in-the-loop approval flow">

**What this covers:** how `interrupt_on` pauses an agent mid-run so a human can approve, edit, or reject a specific tool call before it executes.

**The payoff:** any agent with access to money, message sends, or irreversible actions needs this kind of control before it is used in production.

In the diagram above, every tool call passes through an `Interrupt?` check. If a call matches a rule configured in `interrupt_on`, the agent pauses and hands control to a human instead of executing it directly. The human can approve the call as written, edit its arguments before it runs, or reject it outright, and the agent resumes from exactly where it paused.

This pause only works because a `checkpointer` is attached to the agent: it saves the agent's state at the interrupt point so the run can be resumed later, potentially after the human has stepped away and come back. Resuming looks like calling `agent.invoke` again with `Command(resume={"decision": "approve"})` (or `"edit"` / `"reject"`), rather than starting a new conversation from scratch.

## 5: write_todos (optional), make the plan visible

<img src="./images/deepAgentMiddleware.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Deep Agents middleware stack including TodoList">

**What this covers:** `write_todos`, an opt-in planning tool (enabled via `TodoListMiddleware`) that breaks a multi-step task into a visible, trackable list.

**The payoff:** a visible plan makes a long-running agent's behavior easier to debug and trust.

Once `TodoListMiddleware` is added, the agent gets a `write_todos` tool alongside its filesystem tools, shown in the diagram above. Instead of silently deciding what to do next, the agent writes out a numbered list of steps, then checks items off, or adds new ones, as it works through a task. That list stays visible in the agent's messages the whole time, not just in a final summary.

This matters most once a task needs more than one or two tool calls: without a visible plan, you can only piece together an agent's reasoning after the run finishes, if at all. With `write_todos`, you can watch the plan change step by step and catch a wrong approach early, before it costs several tool calls.

## Wrap-up

You built: a filesystem-backed agent, a swappable role, and a custom tool.

Also covered: human-in-the-loop gating and visible planning (`write_todos`, opt-in via `TodoListMiddleware`). Both can be added with a single argument.

Not covered today, but in the full LangChain Academy Deep Agents course: subagent delegation, backends (filesystem/store/composite), skills, memory across sessions, sandboxes, deployment.